# 15 — Search v2: release years, and the "why" (Steps 9 and 11)

How the app is used (Adil, 2026-10-10): the user types a prompt, a sample movie plus the vibes they want. Search is the main flow.

**Part A (this part): older films should not be the top pick.**
- *Problem.* For "something like How to Lose a Guy in 10 Days" (2003), nothing in search looks at the year, so a 1971 film can come out on top.
- *Candidates*, each an extra equal vote in the existing rank fusion:
  1. **closeness:** year closeness to the sample movie, as part of "how close is this film to the sample";
  2. **stage2:** the same year closeness, as a vote in the popularity reorder;
     - **stage2_older:** the same, but only films *older* than the sample are pushed down (added after the first results showed stage2 pulling old samples toward old films);
  3. **age:** what present-day viewers watch, by film age (no sample movie needed);
  4. combinations of these.
- *Evidence first:* both curves are measured on ratings before the 2018 cutoff (section 1).
- *Test:*
  - a new "more like this" benchmark: for well-known films, the right answers are films liked by the same viewers in the 2018–2019 test period;
  - then Adil's 30 requests: the expected film, and how old the top 10 are.

  A variant is kept only if it beats today's search on the benchmark and does not hurt the requests.

In [1]:
import json
import re
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import boto3
import numpy as np
import pandas as pd
import pyarrow.csv as pc
import torch
from sentence_transformers import SentenceTransformer

ROOT = Path("..")
sys.path.insert(0, str(ROOT.resolve()))
from src.viberec import llm_budget as lb, llm_parse as lp, ranker as rk, request as rq, retrieval as rt, years as yr
from src.viberec.labels import norm
from src.viberec.search_engine import SearchEngine

P, A = ROOT / "data" / "processed", ROOT / "artifacts"
SEED = 42
rng = np.random.default_rng(SEED)


def write_metadata(path, **info):
    meta = {"created_at": datetime.now(timezone.utc).isoformat(), "produced_by": "notebooks/15_search_v2.ipynb", **info}
    path.with_name(path.stem + ".metadata.json").write_text(json.dumps(meta, indent=2, default=str))


early = pd.read_parquet(A / "q43_early_counts_v1.parquet")
VAL_CUT = int(early.loc[early["quantile"] == 0.8, "cutoff_ts"].iloc[0])
TEST_CUT = int(early.loc[early["quantile"] == 0.9, "cutoff_ts"].iloc[0])
CUT_YEAR = pd.to_datetime(TEST_CUT, unit="s").year
tiers = pd.read_parquet(P / "movie_tiers_v2.parquet").set_index("movieId")
year = tiers["year"]
pre = pd.read_parquet(P / "ratings_before_2018_v1.parquet")
print(f"test cutoff {pd.to_datetime(TEST_CUT, unit='s').date()}; ratings before it: {len(pre):,}")

test cutoff 2018-01-04; ratings before it: 22,500,084


## 1. The two curves, measured on ratings before the cutoff

- **Year-gap lift:** take two films the same person liked (rated above their own average) and compare their release years with two random liked films. The result is how much more often the same person likes films g years apart.
- **Age lift:** in the 18.5 months before the cutoff, each film age's share of ratings divided by its share of the tier-3 catalog.

A lift above 1 means more than chance, below 1 means less.

In [2]:
GAP_PATH, AGE_PATH = A / "year_gap_lift_v1.csv", A / "age_watch_lift_v1.csv"
if GAP_PATH.exists() and AGE_PATH.exists():
    print("curves already saved, loading.")
    gap_lift = pd.read_csv(GAP_PATH, index_col=0)["lift"]
    age_lift = pd.read_csv(AGE_PATH, index_col=0)["lift"]
else:
    r = pre[pre["movieId"].isin(year.dropna().index)]
    liked = r[r["rating"] > r.groupby("userId")["rating"].transform("mean")][["userId", "movieId"]]
    liked = liked.assign(year=liked["movieId"].map(year).to_numpy())
    within, random = yr.liked_pair_gaps(liked, n_users=30000, pairs_per_user=40, seed=SEED)
    gap_lift = yr.lift_by_value(within, random, max_value=60).rename("lift")
    recent_r = r[(r["timestamp"] >= VAL_CUT) & (r["timestamp"] < TEST_CUT)]
    age_lift = yr.lift_by_value(CUT_YEAR - recent_r["movieId"].map(year), CUT_YEAR - tiers.loc[tiers["tier"] == 3, "year"],
                                max_value=60).rename("lift")
    gap_lift.to_csv(GAP_PATH); age_lift.to_csv(AGE_PATH)
    write_metadata(GAP_PATH, pairs=len(within), users=30000, note="year gap between two films liked by one user vs two random liked films; ratings before 2018-01-04; 60 = 60+")
    write_metadata(AGE_PATH, cut_year=CUT_YEAR, note="share of ratings in the 18.5 months before the cutoff / share of tier-3 catalog, by film age; 60 = 60+")
show = [0, 1, 2, 3, 5, 8, 12, 17, 25, 35, 50, 60]
print("year gap -> lift:", {g: round(gap_lift[g], 2) for g in show})
print("film age -> lift:", {a: round(age_lift[a], 2) for a in show})
print(f"gap lift falls with every year up to 15: {bool((gap_lift.loc[:15].diff().dropna() < 0).all())}")

curves already saved, loading.
year gap -> lift: {0: np.float64(3.55), 1: np.float64(2.64), 2: np.float64(1.71), 3: np.float64(1.34), 5: np.float64(1.07), 8: np.float64(0.83), 12: np.float64(0.74), 17: np.float64(0.67), 25: np.float64(0.77), 35: np.float64(0.64), 50: np.float64(0.51), 60: np.float64(0.36)}
film age -> lift: {0: np.float64(0.0), 1: np.float64(0.4), 2: np.float64(1.18), 3: np.float64(0.91), 5: np.float64(1.1), 8: np.float64(1.38), 12: np.float64(1.46), 17: np.float64(2.28), 25: np.float64(1.37), 35: np.float64(0.75), 50: np.float64(0.34), 60: np.float64(0.19)}
gap lift falls with every year up to 15: False


## 2. The "more like this" benchmark

- **Queries:** films liked by at least 20 viewers in the test period (from 2018-01-04). 500 of them are drawn at random, and each query is "something like [film]", with nothing else asked.
- **"Liked"** means rated above that viewer's own average.
- **Right answers:** films that the query's fans like **more than expected**. For each film B, the expected number of the query's fans who like B is (all of B's fans) × (the share of all likes that came from the query's fans), which accounts for how much those fans like films in general. A film counts as right when the observed count is clearly higher: a one-sided Poisson test, kept at a 5% false-discovery rate (Benjamini–Hochberg) across all films for that query. This tests similarity, not popularity.
  - v1 counted any film liked by 2 of the fans as right: a median of 4,249 right answers per query, near the ceiling (0.87), so it could not tell the variants apart.
  - v2 compared against all viewers' like rate. But fans of any film are heavy viewers, so nearly everything passed (median 5,883).
  - Neither is used.
- Search runs as it would have on the cutoff day: present-day popularity counts the 18.5 months before the cutoff, and ALS and the age curve also use pre-cutoff ratings only.

In [3]:
encoder = SentenceTransformer("BAAI/bge-small-en-v1.5", device="mps" if torch.backends.mps.is_available() else "cpu")
engine = SearchEngine(ROOT, encoder, tiers=(2, 3))
engine.recent = np.log1p(rk.recent_counts(pre, TEST_CUT, TEST_CUT - VAL_CUT).reindex(engine.ids).fillna(0).to_numpy())
engine.age_curve, engine.now_year = age_lift, CUT_YEAR

from scipy.stats import poisson


def bh_keep(p, alpha=0.05):
    """Benjamini-Hochberg: which p-values are discoveries at false-discovery rate alpha."""
    order = np.argsort(p)
    passed = p[order] <= alpha * np.arange(1, len(p) + 1) / len(p)
    keep = np.zeros(len(p), bool)
    if passed.any():
        keep[order[: np.max(np.where(passed)[0]) + 1]] = True
    return keep


BENCH = A / "more_like_this_bench_v3.parquet"
if BENCH.exists():
    print(f"{BENCH} already exists, loading.")
    bench = pd.read_parquet(BENCH)
else:
    allr = pc.read_csv(ROOT / "data" / "raw" / "movielens" / "ml-25m" / "ratings.csv").to_pandas()
    t = allr[allr["timestamp"] >= TEST_CUT]
    t = t[t["rating"] > t.groupby("userId")["rating"].transform("mean")]
    t = t[t["movieId"].isin(engine.pos.index)]
    likers = t.groupby("movieId")["userId"].apply(set)
    activity = t.groupby("userId").size()                                    # how many films each viewer liked
    n_likers = likers.map(len)
    eligible = likers.index[likers.map(len) >= 20].to_numpy()
    queries = rng.choice(eligible, min(500, len(eligible)), replace=False)
    by_user = t.groupby("userId")["movieId"].apply(list)
    rows = []
    for q in queries:
        co = pd.Series([m for u in likers[q] for m in by_user[u] if m != q]).value_counts()
        # expected likes of B among q's fans, if they liked films like any viewers with the same activity:
        # B's fans in total x the share of all likes that came from q's fans
        expected = n_likers.reindex(co.index).to_numpy() * activity.reindex(list(likers[q])).sum() / activity.sum()
        pval = poisson.sf(co.to_numpy() - 1, expected)
        rows.append({"query": int(q), "likers": len(likers[q]), "relevant": co.index[bh_keep(pval)].astype(int).tolist()})
    bench = pd.DataFrame(rows)
    bench.to_parquet(BENCH, index=False)
    write_metadata(BENCH, queries=len(bench), eligible=len(eligible), viewers=len(activity),
                   rule="liked = above own mean in the test period; query liked by >= 20; relevant = films the query's fans like "
                        "more than expected from their activity (one-sided Poisson, expected = fans of B x share of all likes "
                        "made by the query's fans; Benjamini-Hochberg 5%)")
bench = bench[bench["relevant"].map(len) > 0].reset_index(drop=True)
print(f"{len(bench)} queries with at least one right answer; median {int(bench['relevant'].map(len).median())} right answers each; "
      f"query years: median {int(year.reindex(bench['query']).median())}, "
      f"{(year.reindex(bench['query']) < 1990).mean():.0%} before 1990")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

../artifacts/more_like_this_bench_v3.parquet already exists, loading.
500 queries with at least one right answer; median 555 right answers each; query years: median 2004, 25% before 1990


In [4]:
VARIANTS = {"today": dict(), "closeness": dict(year_mode="closeness"), "stage2": dict(year_mode="stage2"), "stage2_older": dict(year_mode="stage2_older"),
            "age": dict(age=True), "closeness+age": dict(year_mode="closeness", age=True), "stage2+age": dict(year_mode="stage2", age=True)}
rows = []
for b in bench.itertuples():
    parsed = {"reference_ids": [b.query], "genres": [], "vibe_phrases": [], "literal_phrases": [], "vibe_up": [], "vibe_down": []}
    rel = set(b.relevant)
    for name, kw in VARIANTS.items():
        top = engine.rank(parsed, k=10, **kw)
        gaps = np.abs(year.reindex(top).to_numpy() - year[b.query])
        rows.append({"query": b.query, "variant": name, "precision@10": len(set(top) & rel) / 10,
                     "median_gap": np.nanmedian(gaps), "gap_25plus": np.nanmean(gaps >= 25),
                     "before_1990": np.nanmean(year.reindex(top).to_numpy() < 1990)})
mlt = pd.DataFrame(rows)
mlt.to_csv(A / "more_like_this_eval_v3.csv", index=False)
summary = mlt.groupby("variant")[["precision@10", "median_gap", "gap_25plus", "before_1990"]].mean().reindex(list(VARIANTS))
print(summary.round(3).to_string())
base = mlt[mlt["variant"] == "today"].set_index("query")["precision@10"]
for name in list(VARIANTS)[1:]:
    d = (mlt[mlt["variant"] == name].set_index("query")["precision@10"] - base).to_numpy()
    boot = [d[rng.integers(0, len(d), len(d))].mean() for _ in range(2000)]
    print(f"{name:14s} vs today: {d.mean() / base.mean():+.1%}  [{np.percentile(boot, 2.5):+.4f}, {np.percentile(boot, 97.5):+.4f}]")

               precision@10  median_gap  gap_25plus  before_1990
variant                                                         
today                 0.665       5.995       0.083        0.246
closeness             0.652       1.292       0.020        0.239
stage2                0.690       2.256       0.017        0.246
stage2_older          0.645       5.043       0.056        0.155
age                   0.609       7.265       0.077        0.143
closeness+age         0.627       1.269       0.019        0.220
stage2+age            0.657       3.304       0.030        0.192
closeness      vs today: -1.9%  [-0.0292, +0.0038]
stage2         vs today: +3.9%  [+0.0152, +0.0372]
stage2_older   vs today: -2.9%  [-0.0306, -0.0078]
age            vs today: -8.3%  [-0.0652, -0.0458]
closeness+age  vs today: -5.6%  [-0.0554, -0.0194]
stage2+age     vs today: -1.2%  [-0.0184, +0.0020]


## 3. Adil's 30 requests

The same variants on Adil's requests, using Claude's saved readings of them (no new cost). Search runs with the latest popularity, as the app would. The age vote uses the 2018 curve, with "now" set to 2019, the end of MovieLens.
- **Must not get worse:** whether the expected film appears.
- **Should improve:** fewer old films in the top 10.

In [5]:
live = SearchEngine(ROOT, encoder, tiers=(2, 3))
live.age_curve, live.now_year = age_lift, 2019
guard = lb.BudgetGuard(A / "llm_spend_ledger.json", cap_usd=15.0)
client = boto3.Session(profile_name="viberec", region_name="us-east-2").client("bedrock-runtime")
spent0 = guard.spent()
reqs = pd.read_excel(ROOT / "data" / "labels" / "search_requests_v1.xlsx", sheet_name="Requests").dropna(subset=["Your request (as you would type it)"])
reqs.columns = ["n", "request", "expected"]
ml = pd.read_csv(ROOT / "data" / "raw" / "movielens" / "ml-25m" / "movies.csv")
any_index = rq.build_title_index(ml, set(ml["movieId"]), live.tiers["rating_count"].to_dict())
rows = []
for r in reqs.itertuples():
    parsed = live.from_llm(lp.parse(client, guard, r.request, live.attr_tags, ROOT / "data" / "raw" / "bedrock" / "search_parse_v1"))
    exp = any_index.get(norm(re.sub(r"\s*\(\d{4}\)\s*$", "", str(r.expected)))) if isinstance(r.expected, str) else None
    for name, kw in VARIANTS.items():
        top = live.rank(parsed, k=50, vibe_mode="llm_attrs", **kw)
        y10 = year.reindex(top[:10]).to_numpy()
        rows.append({"n": r.n, "variant": name, "expected_in_top10": exp in top[:10] if exp else np.nan,
                     "expected_in_top50": exp in top if exp else np.nan, "before_1990_top10": np.nanmean(y10 < 1990),
                     "median_year_top10": np.nanmedian(y10), "first": live.tiers["title"].get(top[0]) if top else None})
req_eval = pd.DataFrame(rows)
req_eval.to_csv(A / "search_years_requests_eval_v1.csv", index=False)
print(f"no new LLM cost: {guard.spent() - spent0:.4f}")
print(req_eval.groupby("variant")[["expected_in_top10", "expected_in_top50", "before_1990_top10", "median_year_top10"]]
      .mean().reindex(list(VARIANTS)).round(3).to_string())

no new LLM cost: 0.0062
              expected_in_top10 expected_in_top50  before_1990_top10  median_year_top10
variant                                                                                
today                  0.464286          0.678571              0.090           2004.317
closeness              0.392857          0.428571              0.087           2004.100
stage2                 0.428571          0.678571              0.057           2003.717
stage2_older           0.535714          0.678571              0.023           2007.500
age                    0.392857          0.678571              0.020           2003.167
closeness+age          0.392857          0.428571              0.023           2004.050
stage2+age             0.357143          0.678571              0.020           2003.983


### Part A result and decision (Adil, 2026-10-10): **stage2_older**

| Variant | "More like this" benchmark (precision@10) | Adil's requests: expected in top 10 | Pre-1990 films in top 10 | Median year, top 10 |
|---|---|---|---|---|
| today | 0.661 | 46% | 9.3% | 2004 |
| stage2 | **0.684** (+3.5%, interval +0.013 to +0.033) | 43% | 5.3% | 2004 |
| **stage2_older** | 0.639 (−3.3%, −0.031 to −0.012) | **54%** | **2.7%** | **2006** |
| age, combinations | 0.61–0.64 (−3% to −9%) | 36–39% | 2.3% | 2003 |

- **The data:** one person's liked films cluster by year (3.5× more often the same year, 1.3× at 3–4 years, 0.5× at 50+). Present-day viewers watch 1993–2005 films most (1.8×) and 50+-year-old films least (0.2×).
- **The trade-off:** pushing down films far in year from the sample (stage2) best matches MovieLens fans' behaviour. But for an old sample it pulls in old films (Jurassic Park → Independence Day). Pushing down only *older* films (stage2_older) is best on Adil's requests and nearly removes old films, at a 3.3% cost on the MovieLens benchmark.
- **Decision:** stage2_older, Adil's choice. His requests are the target use, and the benchmark reflects 2018 MovieLens habits. The requests evidence is 2 extra hits out of 28, so it should be re-checked on new requests.
- Search setting from now on: `SEARCH_SETTINGS` in `src/viberec/search_engine.py`.

## Part B: the "why" for search results (Step 11, Adil's rules in PROJECT.md §9)

Search fuses up to three parts:
- **vibe:** the feel you asked for;
- **keywords:** literal words from the request, found in the film's story;
- **sample movie:** closeness to it, through the same feel and through viewers with similar taste.

Each result's share of each part is exact: in the fusion, part p adds 1 / (60 + the film's rank in p). The "why" lists its reasons in that order, so it leads with what really put the film there.

**What the "why" may use:**
- the film's genres;
- its feel: its strongest vibes, plus the Light↔Heavy and Pacing labels;
- the vibes you asked for that it has;
- the request's words found in its story;
- the feel it shares with the sample movie;
- "viewers with similar taste rate it highly";
- "well known".

Spoiler vibes (happy ending, twist, tragedy, non-Hollywood ending) and the Ending label are never used.

**Tests**
1. **Is the stated lead reason real?** For every top-10 result of requests that use 2 or more parts, remove each part in turn and see how far the film falls. Removing the lead part should hurt the most.
2. **Grounding and spoilers (automatic):** every film, vibe and genre in Claude's sentence must come from the evidence, and no ending words are allowed.
3. **Adil's blind sheet:** for each request, the #1 result with two "why"s (free template vs Claude).

Cost: Claude words the top 3 of each request, about 90 calls, roughly $0.10.

**Version 2 (2026-10-10).** In v1 the evidence had only feel words and genres, so Claude filled the gaps from memory, sometimes wrongly ("The Martian explores consciousness and mind manipulation") or with a soft spoiler ("a sweet resolution"). v2 adds each film's **TMDB premise**, the short overview written to sell the film, cut to about 60 words. Claude may describe the setup only from it. The checker also became fairer: titles count only when written as titles, and words from the premise count as evidence.

In [6]:
from src.viberec import explain as ex
from src.viberec.search_engine import SEARCH_SETTINGS

three = pd.read_parquet(P / "vibe_three_v1.parquet").set_index("movieId")
S = SEARCH_SETTINGS
parsed_reqs = {r.n: (r.request, live.from_llm(lp.parse(client, guard, r.request, live.attr_tags,
                                                        ROOT / "data" / "raw" / "bedrock" / "search_parse_v1")))
               for r in reqs.itertuples()}
check = np.mean([any_index.get(norm(re.sub(r"\s*\(\d{4}\)\s*$", "", str(e)))) in
                 live.rank(parsed_reqs[n][1], k=10, vibe_mode=S["vibe_mode"], year_mode=S["year_mode"])
                 for n, e in zip(reqs["n"], reqs["expected"])
                 if isinstance(e, str) and any_index.get(norm(re.sub(r"\s*\(\d{4}\)\s*$", "", e))) is not None])
print(f"same search as Part A's decision after the code change: expected film in top 10 {check:.1%} (Part A: 53.6%)")

# Test 1: remove each part in turn
rows = []
for n, (text, parsed) in parsed_reqs.items():
    parts, _ = live.relevance_parts(parsed, S["vibe_mode"])
    if len(parts) < 2:
        continue
    top = live.rank(parsed, k=10, vibe_mode=S["vibe_mode"], year_mode=S["year_mode"])
    without = {p_: live.rank(parsed, k=live.n_relevant, vibe_mode=S["vibe_mode"], year_mode=S["year_mode"], skip_parts=(p_,))
               for p_ in parts}
    for old_rank, m in enumerate(top, 1):
        contrib = ex.part_contributions(parts, int(live.pos[m]))
        for p_, lst in without.items():
            new_rank = lst.index(m) + 1 if m in lst else live.n_relevant + 1
            rows.append({"n": n, "movieId": m, "part": p_, "is_lead": p_ == contrib.index[0], "parts": len(parts),
                         "drop": new_rank - old_rank})
deletion = pd.DataFrame(rows)
deletion.to_csv(A / "search_why_deletion_v1.csv", index=False)
per_movie = deletion.groupby(["n", "movieId"])
lead_biggest = per_movie.apply(lambda g: g.loc[g["is_lead"], "drop"].iloc[0] >= g["drop"].max()).mean()
chance = (1 / deletion.groupby(["n", "movieId"])["parts"].first()).mean()
print(f"{deletion[['n', 'movieId']].drop_duplicates().shape[0]} results from {deletion['n'].nunique()} requests with 2+ parts")
print("places fallen when a part is removed:", deletion.groupby("is_lead")["drop"].mean().rename({True: "lead part", False: "other parts"}).round(1).to_dict())
print(f"removing the lead part hurts the most for {lead_biggest:.0%} of results (chance would be {chance:.0%})")

same search as Part A's decision after the code change: expected film in top 10 53.6% (Part A: 53.6%)


300 results from 30 requests with 2+ parts
places fallen when a part is removed: {'other parts': 0.4, 'lead part': 36.2}
removing the lead part hurts the most for 93% of results (chance would be 39%)


In [7]:
from src.viberec import tmdb, wikipedia as wp

CACHE = ROOT / "data" / "raw" / "bedrock" / "search_why_v1"
TMDB_DIR = ROOT / "data" / "raw" / "tmdb" / "movie_v1"
tmdb_of = live.tiers["tmdbId"]

# the premise: TMDB's short overview (written to sell the film, not to tell the ending), fetched only if not cached
results = {m for n, (text, parsed) in parsed_reqs.items()
           for m in live.rank(parsed, k=3, vibe_mode=S["vibe_mode"], year_mode=S["year_mode"])}
need = {int(tmdb_of[m]): int(tmdb_of[m]) for m in results if pd.notna(tmdb_of.get(m))}
key = tmdb.api_key()
fetched, skipped = wp.fetch_each(need, TMDB_DIR, lambda t: tmdb.fetch_movie(t, key), pause=0.1)
print(f"TMDB premises for {len(results)} result films: {fetched} fetched now, {skipped} already cached")


def premise(m):
    f = TMDB_DIR / f"{int(tmdb_of[m])}.json" if pd.notna(tmdb_of.get(m)) else None
    return ex.premise_text(json.loads(f.read_text()).get("overview")) if f is not None and f.exists() else ""


catalog = list(live.tiers.loc[live.ids].sort_values("rating_count", ascending=False)["title"].head(5000))
spent0 = guard.spent()
rows = []
for n, (text, parsed) in parsed_reqs.items():
    parts, sub = live.relevance_parts(parsed, S["vibe_mode"])
    top = live.rank(parsed, k=3, vibe_mode=S["vibe_mode"], year_mode=S["year_mode"])
    for pos_, m in enumerate(top, 1):
        e = ex.search_evidence(live, parsed, text, m, parts, sub, three=three, premise=premise(m))
        words = set(e["feel"]) | {w for r in e["reasons"] for k in ("has", "is_not", "shared_feel") for w in r.get(k, [])}
        allowed_vibes = [a for a, w in ex.VIBE_WORDS.items() if w in words]
        allowed_titles = [e["movie"]] + [r["sample_movie"] for r in e["reasons"] if "sample_movie" in r]
        claude = ex.search_llm_wording(client, guard, lp.MODEL_ID, e, CACHE)
        rows.append({"n": n, "request": text, "position": pos_, "movieId": m, "movie": f"{e['movie']}",
                     "lead": e["reasons"][0]["reason"], "template": ex.search_template(e), "claude": claude,
                     "problems": "; ".join(ex.grounding_problems(claude, allowed_titles, allowed_vibes, catalog,
                                                                 allowed_genres=e["genres"] + parsed["genres"],
                                                                 evidence_text=e["premise"])
                                           + ex.spoiler_problems(claude)),
                     "template_problems": "; ".join(ex.spoiler_problems(ex.search_template(e)))})
why = pd.DataFrame(rows)
why.to_csv(A / "search_why_wording_v2.csv", index=False)
print(f"Claude cost ${guard.spent() - spent0:.4f}; total spend ${guard.spent():.4f}; "
      f"results with a premise: {(why['movieId'].map(premise) != '').mean():.0%}")
print(f"Claude sentences with an ungrounded film/vibe/genre or a spoiler word: {(why['problems'] != '').sum()} of {len(why)}; "
      f"template sentences with a spoiler word: {(why['template_problems'] != '').sum()}")
for r in why[why["problems"] != ""].itertuples():
    print(f"- #{r.n} {r.movie}: {r.claude}\n    flagged: {r.problems}")
for r in why[why["position"] == 1].head(6).itertuples():
    print(f"\n#{r.n} {r.request}\n -> {r.movie}\n  template: {r.template}\n  Claude:   {r.claude}")

TMDB premises for 81 result films: 26 fetched now, 55 already cached


Claude cost $0.0326; total spend $9.2403; results with a premise: 100%
Claude sentences with an ungrounded film/vibe/genre or a spoiler word: 13 of 90; template sentences with a spoiler word: 3
- #1 Captain America: Civil War (2016): This fast-paced, visually stunning action-thriller delivers the same witty superhero energy as Iron Man, with polarized Avengers factions clashing in epic battles. Its strong ensemble cast and high-stakes conflict create the commanding character dynamics you're seeking.
    flagged: title: Avengers, The (2012); title: Avengers, The (1998)
- #3 Ted (2012): This funny fantasy comedy uses slapstick satire to explore an absurd premise—a man navigating life with his talking teddy bear. It matches your taste for sharp humor and satirical comedy while delivering the same comedic energy as 21 Jump Street, just with a fantastical twist.
    flagged: spoiler: twist
- #3 The Hangover (2009): This light, witty comedy-crime film follows three friends scrambling through

### Adil's blind sheet

For each of Adil's requests, the #1 result with two "why"s. A and B are in random order; which one is Claude's is in a separate key file. Sheet: `data/labels/search_why_v2.xlsx`.

In [8]:
SHEET = ROOT / "data" / "labels" / "search_why_v2.xlsx"
KEY = ROOT / "data" / "labels" / "search_why_v2_key.parquet"
if SHEET.exists():
    print(f"{SHEET} already exists; not overwritten (it may hold Adil's answers).")
else:
    first = why[why["position"] == 1].reset_index(drop=True)
    claude_is_a = rng.random(len(first)) < 0.5
    sheet = pd.DataFrame({
        "#": first["n"], "Your request": first["request"], "Top result": first["movie"],
        "Why A": np.where(claude_is_a, first["claude"], first["template"]),
        "Why B": np.where(claude_is_a, first["template"], first["claude"]),
        "Which is better? (A / B / same)": "", "Is either wrong? (A / B / both / neither)": "",
        "Does either spoil the movie? (A / B / both / neither)": "", "Notes": ""})
    howto = pd.DataFrame({"How to fill this in": [
        "Each row is one of your own search requests, with the top movie the app found for it.",
        "Imagine you typed that request. Under the movie you would see one short 'why'. Columns D and E are two versions.",
        "One is a fill-in-the-blank sentence and the other was written by Claude; which is which is hidden.",
        "Column F: which 'why' would you rather see? Type A, B or same.",
        "Column G: does either say something wrong about the movie or your request? Type A, B, both or neither.",
        "Column H: does either give away the story (ending, twist, who dies)? Type A, B, both or neither.",
        "Notes are optional. Save the file when done."]})
    with pd.ExcelWriter(SHEET) as w:
        howto.to_excel(w, sheet_name="How to", index=False)
        sheet.to_excel(w, sheet_name="Why", index=False)
    first.assign(claude_is=np.where(claude_is_a, "A", "B"))[["n", "movieId", "claude_is"]].to_parquet(KEY, index=False)
    print(f"saved {SHEET} ({len(sheet)} rows) and the key")

../data/labels/search_why_v2.xlsx already exists; not overwritten (it may hold Adil's answers).


## Part C: comparison of search designs (Adil, 2026-10-10)

Adil's design is two steps:
1. **Neighbourhood:** collect the N films closest to the sample movie (same feel + same fans).
2. **Prompt picks:** rank only those N by what the prompt adds (vibes, keywords), with closeness, familiarity and "not older than the sample" as equal votes.

**Designs compared** (the parser, vibe scores, ALS, tiers and popularity are the same in all of them):
- **current:** one fusion over the whole catalog, then the popularity / year reorder (Part A's decision);
- **two_step_N:** Adil's design with N = 20, 30, 50 or 100 neighbours. The size is decided here by the test, not chosen by hand;
- **two_step_learned_N:** the same, but closeness to the sample is a **logistic regression** over four factors (same feel, same fans, present-day popularity, years older than the sample). It is fitted on half of the "more like this" benchmark films and tested on the other half, so the weights come from data and are shown as an equation.

How much the prompt's extra vibes should weigh cannot be learned yet: 28 requests are far too few. They keep an equal vote.

**Measures**
- benchmark (the 250 held-out films): right answers in the top 10;
- Adil's requests: expected film in the top 10, familiarity (median ratings of the top 10), pre-1990 share;
- **stays near the sample:** the share of the top 10 that is among the sample's 50 closest films.

In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from src.viberec.search_engine import NEIGHBOUR_FEATURES as F, neighbour_features, rank_two_step

rel = dict(zip(bench["query"], bench["relevant"].map(set)))
perm = rng.permutation(len(bench))
train_q, test_q = bench["query"].to_numpy()[perm[: len(bench) // 2]], bench["query"].to_numpy()[perm[len(bench) // 2:]]


def candidates(eng, q, n=300):
    """The n films closest to q by the equal vote, with their features and whether they are a right answer."""
    rows = np.arange(len(eng.ids))[eng.ids != q]
    f = neighbour_features(eng, q, rows)
    top = np.argsort(-rt.rrf([f["same_feel"].to_numpy(), f["similar_taste"].to_numpy()]))[:n]
    f = f.iloc[top].assign(query=q, movieId=eng.ids[rows][top])
    return f.assign(label=f["movieId"].map(lambda m: m in rel[q]).astype(int))


train_rows = pd.concat([candidates(engine, q) for q in train_q], ignore_index=True)
X, y, g = train_rows[F].to_numpy(), train_rows["label"].to_numpy(), train_rows["query"].to_numpy()
scaler = StandardScaler().fit(X)
cv = {}
for C in (0.001, 0.01, 0.1, 1.0, 10.0):                                   # penalty strength chosen by grouped CV
    ll = []
    for tr, te in GroupKFold(5).split(X, y, g):
        m = LogisticRegression(C=C, max_iter=2000).fit(scaler.transform(X[tr]), y[tr])
        p_ = m.predict_proba(scaler.transform(X[te]))[:, 1]
        ll.append(-np.mean(y[te] * np.log(p_ + 1e-12) + (1 - y[te]) * np.log(1 - p_ + 1e-12)))
    cv[C] = np.mean(ll)
best_C = min(cv, key=cv.get)
model = LogisticRegression(C=best_C, max_iter=2000).fit(scaler.transform(X), y)
learned = (scaler, model)
print(f"{len(train_q)} training films, {len(train_rows):,} candidate rows, {y.mean():.1%} right answers; "
      f"C chosen {best_C} (cross-validated log loss {cv})")
coef = pd.Series(model.coef_[0], index=F)
print("learned equation (features standardised; a bigger weight counts more):")
print("  score = " + " + ".join(f"{w:+.2f} x {n}" for n, w in coef.items()) + f" {model.intercept_[0]:+.2f}")

250 training films, 75,000 candidate rows, 30.4% right answers; C chosen 0.01 (cross-validated log loss {0.001: np.float64(0.49594135126048755), 0.01: np.float64(0.49457749920075444), 0.1: np.float64(0.4945964764103987), 1.0: np.float64(0.4946008051434335), 10.0: np.float64(0.4946012636548597)})
learned equation (features standardised; a bigger weight counts more):
  score = +0.12 x same_feel + +0.52 x similar_taste + +1.17 x log_recent + -0.11 x years_older -1.15


In [10]:
DESIGNS = {"current": None, **{f"two_step_{n}": (n, None) for n in (20, 30, 50, 100)},
           **{f"two_step_learned_{n}": (n, learned) for n in (20, 30, 50, 100)}}


def run(eng, parsed, design, k=10):
    if DESIGNS[design] is None:
        return eng.rank(parsed, k=k, vibe_mode="llm_attrs", year_mode="stage2_older")
    n, lm = DESIGNS[design]
    return rank_two_step(eng, parsed, k=k, n_neighbours=n, learned=lm)


def near_sample(eng, refs, top, n=50):
    if not refs or not top:
        return np.nan
    near = set()
    for ref in refs:
        f = neighbour_features(eng, ref)
        order = np.argsort(-rt.rrf([f["same_feel"].to_numpy(), f["similar_taste"].to_numpy()]))
        near |= set(eng.ids[order[: n + 1]]) - {ref}
    return np.mean([m in near for m in top])


rows = []
for q in test_q:
    parsed = {"reference_ids": [q], "genres": [], "vibe_phrases": [], "literal_phrases": [], "vibe_up": [], "vibe_down": []}
    for d in DESIGNS:
        top = run(engine, parsed, d)
        rows.append({"query": q, "design": d, "precision@10": len(set(top) & rel[q]) / 10,
                     "median_ratings": live.tiers["rating_count"].reindex(top).median(),
                     "before_1990": np.nanmean(year.reindex(top).to_numpy() < 1990), "near_sample": near_sample(engine, [q], top)})
cmp_bench = pd.DataFrame(rows)
cmp_bench.to_csv(A / "search_design_bench_v1.csv", index=False)
base = cmp_bench[cmp_bench["design"] == "current"].set_index("query")["precision@10"]
out = cmp_bench.groupby("design")[["precision@10", "near_sample", "before_1990", "median_ratings"]].mean().reindex(list(DESIGNS))
gain = {}
for d in list(DESIGNS)[1:]:
    diff = (cmp_bench[cmp_bench["design"] == d].set_index("query")["precision@10"] - base).to_numpy()
    boot = [diff[rng.integers(0, len(diff), len(diff))].mean() for _ in range(2000)]
    gain[d] = f"{diff.mean() / base.mean():+.1%} [{np.percentile(boot, 2.5):+.3f}, {np.percentile(boot, 97.5):+.3f}]"
out["vs current"] = pd.Series(gain)
print(f"benchmark, {len(test_q)} held-out films:")
print(out.round(3).to_string())

benchmark, 250 held-out films:
                      precision@10  near_sample  before_1990  median_ratings               vs current
design                                                                                               
current                      0.618        1.000        0.166        2761.598                      NaN
two_step_20                  0.635        1.000        0.205        2456.070   +2.7% [+0.004, +0.028]
two_step_30                  0.628        1.000        0.182        2605.694   +1.5% [+0.002, +0.017]
two_step_50                  0.618        1.000        0.165        2776.686   -0.1% [-0.003, +0.001]
two_step_100                 0.602        0.943        0.144        3242.598   -2.6% [-0.026, -0.007]
two_step_learned_20          0.498        0.409        0.177        8009.830  -19.5% [-0.152, -0.088]
two_step_learned_30          0.498        0.409        0.177        8009.830  -19.5% [-0.152, -0.090]
two_step_learned_50          0.498        0.409    

In [11]:
rows, tops = [], {}
for n, (text, parsed) in parsed_reqs.items():
    exp = any_index.get(norm(re.sub(r"\s*\(\d{4}\)\s*$", "", str(reqs.set_index("n").at[n, "expected"]))))
    for d in DESIGNS:
        top = run(live, parsed, d, k=10)
        tops[(n, d)] = top
        rows.append({"n": n, "design": d, "has_sample": bool(parsed["reference_ids"]),
                     "expected_in_top10": (exp in top) if exp else np.nan,
                     "median_ratings": live.tiers["rating_count"].reindex(top).median(),
                     "before_1990": np.nanmean(year.reindex(top).to_numpy() < 1990),
                     "near_sample": near_sample(live, parsed["reference_ids"], top)})
cmp_req = pd.DataFrame(rows)
cmp_req.to_csv(A / "search_design_requests_v1.csv", index=False)
print(f"Adil's requests: {cmp_req[cmp_req['design'] == 'current']['has_sample'].sum()} of {len(parsed_reqs)} name a sample movie "
      f"(the others run the current search in every design)")
print(cmp_req.groupby("design")[["expected_in_top10", "near_sample", "before_1990", "median_ratings"]].mean().reindex(list(DESIGNS)).round(3).to_string())
for n in (2, 5, 12, 23):
    print(f"\n#{n} {parsed_reqs[n][0]}")
    for d in ("current", "two_step_30", "two_step_learned_30"):
        print(f"  {d:22s} " + " · ".join(ex.display_title(live.tiers.at[m, 'title']) for m in tops[(n, d)][:5]))

Adil's requests: 30 of 30 name a sample movie (the others run the current search in every design)
                     expected_in_top10  near_sample  before_1990  median_ratings
design                                                                          
current                       0.535714        0.727        0.023        9837.933
two_step_20                   0.392857        0.897        0.033       11930.400
two_step_30                   0.428571        0.867        0.013       12447.550
two_step_50                   0.464286        0.827        0.017       12337.533
two_step_100                  0.428571        0.767        0.020       11617.067
two_step_learned_20           0.464286        0.657        0.033       19818.950
two_step_learned_30                0.5        0.650        0.033       18420.433
two_step_learned_50           0.428571        0.613        0.040       16084.867
two_step_learned_100          0.464286        0.583        0.043       13780.683

#2 A light

## Part D: blending the two designs (Adil's idea, 2026-10-10)

The two designs fail differently:
- **two-step** misses films that the prompt's words point to, when they are far from the sample (Rush Hour for White Chicks + "funny duo, action"; Into the Spider-Verse for Amazing Spider-Man + "animated");
- **current** strays further from the sample.

**Blend:** a 5-film list takes the first *a* films from current and fills the other 5 − *a* from two-step (30 neighbours), skipping duplicates. Every split from 5/0 to 0/5 is tested, so the split comes from the data, not by hand.

**Measures:**
- Adil's requests: expected film among the 5;
- benchmark: right answers among the 5 (250 held-out films);
- share of the 5 that is near the sample;
- how many films differ from the pure current list.

In [12]:
from src.viberec.search_engine import blend

SPLITS = [5, 4, 3, 2, 1, 0]                                          # films taken from current
rows = []
for n, (text, parsed) in parsed_reqs.items():
    exp = any_index.get(norm(re.sub(r"\s*\(\d{4}\)\s*$", "", str(reqs.set_index("n").at[n, "expected"]))))
    a, b = tops[(n, "current")], tops[(n, "two_step_30")]
    for s in SPLITS:
        five = blend(a, b, s)
        rows.append({"n": n, "split": f"{s}/{5 - s}", "expected_in_5": (exp in five) if exp else np.nan,
                     "near_sample": near_sample(live, parsed["reference_ids"], five), "differs_from_current": len(set(five) - set(a[:5]))})
blend_req = pd.DataFrame(rows)
rows = []
for q in test_q:
    parsed = {"reference_ids": [q], "genres": [], "vibe_phrases": [], "literal_phrases": [], "vibe_up": [], "vibe_down": []}
    a, b = run(engine, parsed, "current"), run(engine, parsed, "two_step_30")
    for s in SPLITS:
        five = blend(a, b, s)
        rows.append({"query": q, "split": f"{s}/{5 - s}", "precision@5": len(set(five) & rel[q]) / 5})
blend_bench = pd.DataFrame(rows)
order = [f"{s}/{5 - s}" for s in SPLITS]
table = blend_req.groupby("split")[["expected_in_5", "near_sample", "differs_from_current"]].mean().reindex(order)
table["benchmark precision@5"] = blend_bench.groupby("split")["precision@5"].mean().reindex(order)
base = blend_bench[blend_bench["split"] == "5/0"].set_index("query")["precision@5"]
for s in order[1:]:
    d = (blend_bench[blend_bench["split"] == s].set_index("query")["precision@5"] - base).to_numpy()
    boot = [d[rng.integers(0, len(d), len(d))].mean() for _ in range(2000)]
    table.loc[s, "benchmark vs 5/0"] = f"{d.mean() / base.mean():+.1%} [{np.percentile(boot, 2.5):+.3f}, {np.percentile(boot, 97.5):+.3f}]"
exp_hits = blend_req.groupby("split")["expected_in_5"].sum().reindex(order)
table["expected films found (of 28)"] = exp_hits.astype(int)
print("split = films from current / films from two-step")
print(table.round(3).to_string())
pd.concat([blend_req.assign(source="requests"), blend_bench.assign(source="benchmark")]).to_csv(A / "search_blend_eval_v1.csv", index=False)
for n in (2, 7, 12, 16):
    print(f"\n#{n} {parsed_reqs[n][0][:90]}")
    for s in (5, 3, 0):
        print(f"  {s}/{5 - s}: " + " · ".join(ex.display_title(live.tiers.at[m, 'title']) for m in blend(tops[(n, 'current')], tops[(n, 'two_step_30')], s)))

split = films from current / films from two-step
      expected_in_5  near_sample  differs_from_current  benchmark precision@5        benchmark vs 5/0  expected films found (of 28)
split                                                                                                                              
5/0        0.321429        0.787                 0.000                  0.644                     NaN                             9
4/1        0.357143        0.807                 0.767                  0.650  +0.9% [-0.003, +0.015]                            10
3/2        0.357143        0.813                 1.233                  0.654  +1.5% [-0.002, +0.021]                            10
2/3        0.357143        0.873                 1.700                  0.655  +1.7% [+0.001, +0.022]                            10
1/4        0.321429        0.887                 1.900                  0.654  +1.6% [-0.001, +0.022]                             9
0/5        0.285714        

## Results of Parts B–D (2026-10-10)

**Part C, search designs (decision: keep current).**
- Benchmark precision@10: current 0.618; two-step 20/30 neighbours 0.643/0.642 (+4%); 50 same; 100 −2%; learned weights 0.494 (−20%).
- Adil's requests, expected film in top 10: current 15/28, two-step 10–12/28.
- The learned equation: 0.13 × same feel + 0.53 × same fans + 1.22 × popularity − 0.07 × years older. The regression learned "popular = right", because popular films have more statistically confirmed fans, and pulled in famous but unrelated films.
- Two-step cannot reach films the prompt's words point to when they are far from the sample (Rush Hour, Into the Spider-Verse), or films just past the cut (Eternal Sunshine 56th, Pursuit of Happyness 47th).

**Part D, blends (decision: keep current only).**
- Expected films among 5 (of 28): 5/0 12, 4/1 12, 3/2 10, 2/3 10, 1/4 9, 0/5 8.
- Benchmark: all splits within ±2% (4/1 −1.8% significant, the rest not).

**Part B, the "why".**
- **Lead reason is real:** removing the part the "why" leads with drops the film 36 places on average, against 2.5 for the other parts. It is the biggest cause for 89% of results (chance 39%).
- **v2, with the TMDB premise** (90 sentences, $0.07): 15 flagged automatically. Read by hand:
  - 2 real spoilers: "an unexpected twist that redefines everything" (Inception) and "rewards patience with a deeply satisfying twist" (Slumdog Millionaire);
  - 5 mild descriptions not in the evidence: thriller, survival, revenge, witty, dark comedy;
  - 8 false alarms: "death row" is the premise's setup, "a fantastical twist" is a figure of speech, "cosmic mysteries", "the American Revolution" read as the film *The American*.
  - Template sentences: 0 real problems (2 false "death row" flags).
- **Known weakness of the free template:** it is long (median 64 words vs Claude's 40) and clumsy ("A light, marriage, big-city comedy/romance film"), which favours Claude in the blind test.
- **Waiting on Adil's blind ratings:** `data/labels/search_why_v2.xlsx`.